# Pathology Feature Extraction using Phikon-v2
This notebook extracts deep features from pathology images using Owkin's **Phikon-v2** Vision Transformer pretrained with DINOv2. Works for both standard image formats and `.svs` WSI tiles.

In [1]:
import os
import pandas as pd
from PIL import Image
import torch
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModel
import general_fcns as gf

In [2]:


try:
    import openslide
except ImportError:
    openslide = None
    print("openslide not found, .svs files will be skipped")

# Output folder
output_folder = "./output_features"
os.makedirs(output_folder, exist_ok=True)

# Load model and processor
processor = AutoImageProcessor.from_pretrained("owkin/phikon-v2")
model = AutoModel.from_pretrained("owkin/phikon-v2")
model.eval()

# CSV path and skip list
csv_file_path = os.path.join(os.getcwd(), 'filtered_metadata_full_path.csv')
skip = ["D:/Digital_path_unzipped/SZMC1050237_pdl1.ndpi", "D:/Digital_path/DIG_PAT_1727366215.ndpi"]

# Load image paths from CSV
df = pd.read_csv(csv_file_path)
image_paths = df.iloc[:, 0].dropna().tolist()
image_paths = [p for p in image_paths if p not in skip]

# Valid extensions
supported_ext = [".jpg", ".jpeg", ".png", ".tif", ".tiff", ".svs", ".ndpi"]
openslide_exts = [".svs", ".ndpi"]

# Define slide loader
def slide_at_magnification(slide_path, target_magnification=20, tile_size=224):
    slide = openslide.OpenSlide(slide_path)
    try:
        objective_power = float(slide.properties[openslide.PROPERTY_NAME_OBJECTIVE_POWER])
        level_downsample = int(objective_power / target_magnification)
    except:
        level_downsample = 1
    level = slide.get_best_level_for_downsample(level_downsample)
    w, h = slide.level_dimensions[level]
    x = w // 2 - tile_size // 2
    y = h // 2 - tile_size // 2
    region = slide.read_region((x * level_downsample, y * level_downsample), level, (tile_size, tile_size))
    return region.convert("RGB")

# Process images from CSV
for filepath in tqdm(image_paths):
    ext = os.path.splitext(filepath)[-1].lower()
    if ext not in supported_ext:
        continue

    filename = os.path.basename(filepath)

    try:
        if ext in openslide_exts and openslide is not None:
            image = gf.slide_at_magnification(filepath, magnification_params={'magnification': 10})
        else:
            image = Image.open(filepath).convert("RGB")

        inputs = processor(image, return_tensors="pt")

        with torch.inference_mode():
            outputs = model(**inputs)
            features = outputs.last_hidden_state[:, 0, :]  # CLS token












        output_path = os.path.join(output_folder, filename + ".pt")
        torch.save(features, output_path)

    except Exception as e:
        print(f"Failed to process {filename}: {e}")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
 58%|█████▊    | 273/469 [9:26:20<59:42, 18.28s/it]     

Failed to process DIG_PAT_1701719514.svs: No available tilesource for D:/Digital_path/DIG_PAT_1701719514.svs


100%|██████████| 469/469 [17:35:03<00:00, 134.97s/it]    
